# 8주차 중간고사 — 사전학습 모델로 내 사진 분류하기

- 시간 120분 / 총점 100점 / 개인 평가
- **답안은 시험지(구글 문서 사본)의 답안 칸에 씁니다.** 이 노트북은 실습을 하고 결과를 확인하는 곳입니다.
- 5주차 워크북처럼 **실습 1부터 순서대로** 셀을 실행하세요. `____` 로 표시된 빈칸만 채우면 됩니다.
- 빈칸을 채우지 않고 실행하면 `NameError: name '____' is not defined` 오류가 납니다. 빈칸을 채우고 다시 실행하세요.
- 시험 중 모델을 새로 학습하지 않습니다. ImageNet으로 사전학습된 MobileNetV2를 불러와 사용합니다.

**제출:** 답안지(PDF 또는 공유 링크), `학번_이름_midterm.ipynb`, `midterm_prediction_results.csv`

## 실습 1. 라이브러리 불러오기 → 답안지 1-1
셀을 실행하고 출력된 TensorFlow 버전을 답안지 1-1에 적으세요.

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from PIL import Image
from google.colab import files

print('TensorFlow 버전:', tf.__version__)

## 실습 2. 사전학습 MobileNetV2 불러오기 → 답안지 1-2, 1-3
빈칸 `____` 에 `'imagenet'` 을 넣고 실행하세요. (5주차 워크북 실습 5와 같은 방법)
첫 실행 때 가중치 파일을 내려받느라 수십 초 걸릴 수 있습니다.

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions

# TODO: 빈칸에 'imagenet' 을 넣으세요.
model = MobileNetV2(weights=____)

print('Input shape:', model.input_shape)

## 실습 3. 사진 3장 업로드하기 → 답안지 2-1
직접 촬영한 **서로 다른 사물 사진 3장**(JPG 또는 PNG)을 [파일 선택]에서 한 번에 고르세요.
아이폰 HEIC 사진은 업로드되지 않으니 JPG로 저장해 오세요.

In [ ]:
uploaded = files.upload()
image_names = [name for name in uploaded if name.lower().endswith(('.jpg', '.jpeg', '.png'))]
assert len(image_names) >= 3, 'JPG/PNG 사진 3장 이상을 업로드하세요.'

for i, name in enumerate(image_names, 1):
    print(i, name)

## 실습 4. 사진 전처리하기 → 답안지 2-2
빈칸에 `'RGB'` 와 `224` 를 넣고 실행하세요. 출력된 `x.shape`, 최솟값, 최댓값을 답안지에 적습니다.

전처리 순서: 사진 열기 → RGB 변환 → 224×224 크기 변경 → 숫자 배열로 변환 → `preprocess_input`

In [ ]:
def load_image(name):
    image = Image.open(name).convert(____)          # TODO: 'RGB'
    image = image.resize((____, ____))              # TODO: 224, 224
    x = np.asarray(image, dtype=np.float32)[None, ...]   # 사진 1장을 묶음(batch) 형태로
    x = preprocess_input(x)                          # MobileNetV2가 원하는 값 범위로 변환
    return x

x = load_image(image_names[0])
print('x.shape:', x.shape)
print('최솟값:', round(float(x.min()), 3), '/ 최댓값:', round(float(x.max()), 3))

## 실습 5. Top-3 예측하기 → 답안지 2-3
빈칸에 `3` 을 넣고 실행하세요. 사진별 1위~3위 클래스와 확률을 답안지에 적습니다.

In [ ]:
results = []
for name in image_names:
    x = load_image(name)
    preds = model.predict(x, verbose=0)
    top3 = decode_predictions(preds, top=____)[0]   # TODO: 3
    print(f'\n[{name}]')
    for rank, (_, label, prob) in enumerate(top3, 1):
        print(f'  {rank}위: {label} ({prob:.4f})')
    results.append({'file': name, 'top1_label': top3[0][1], 'top1_confidence': round(float(top3[0][2]), 4)})

## 실습 6. 결과표, 정확도, CSV 저장 → 답안지 3-1, 3-2, 3-3
1. `actual` 에 업로드 순서대로 실제 사물 이름을 적으세요. 예: `['컵', '마우스', '운동화']`
2. `correct` 에 Top-1 예측이 맞으면 `1`, 틀리면 `0` 을 적으세요.
3. 실행한 뒤 왼쪽 파일 패널에서 `midterm_prediction_results.csv` 를 내려받으세요.

In [ ]:
CSV = 'midterm_prediction_results.csv'
df = pd.DataFrame(results)
print(df[['file', 'top1_label']])

# TODO: 업로드 순서대로 실제 사물 이름
actual = ['____', '____', '____']
# TODO: Top-1 예측이 맞으면 1, 틀리면 0
correct = [0, 0, 0]

assert len(actual) == len(df) and len(correct) == len(df), '사진 수만큼 actual, correct 를 적으세요.'
df['actual'] = actual
df['correct'] = correct
df['condition'] = '원본'
display(df)

print(f"정확도: 맞힌 수 {df['correct'].sum()} / 전체 수 {len(df)} = {df['correct'].mean() * 100:.1f}%")
df.to_csv(CSV, index=False, encoding='utf-8-sig')
print('저장 완료:', CSV)

## 실습 7. 조건을 바꿔 다시 실험하기 → 답안지 4-1, 4-2
1. 실습 3의 사물 중 하나를 골라 **조명, 배경, 각도 중 한 가지만** 바꿔 다시 촬영하세요.
2. `condition` 에 바꾼 조건, `original_file` 에 비교할 원본 파일명을 적고 실행한 뒤 새 사진을 업로드하세요.
3. 원본과 새 사진의 Top-1 예측과 확률을 비교해 답안지 4-1에 적고, 4-2를 작성하세요.

In [ ]:
condition = '____'       # TODO: 예) '어두운 조명', '복잡한 배경', '위에서 내려다본 각도'
original_file = '____'   # TODO: 비교할 원본 사진 파일명 (실습 3에서 올린 파일 중 하나)
assert original_file in df['file'].values, '원본 파일명을 실습 3에서 출력된 이름 그대로 적으세요.'

new_upload = files.upload()
new_name = [n for n in new_upload if n.lower().endswith(('.jpg', '.jpeg', '.png'))][0]

preds = model.predict(load_image(new_name), verbose=0)
top3 = decode_predictions(preds, top=3)[0]
orig = df[df['file'] == original_file].iloc[0]

print(f"원본 사진 [{original_file}]: {orig['top1_label']} ({orig['top1_confidence']:.4f})")
print(f"변경 사진 [{new_name}] ({condition}): {top3[0][1]} ({float(top3[0][2]):.4f})")

new_correct = 0   # TODO: 변경 사진의 Top-1 예측이 맞으면 1, 틀리면 0
df = pd.concat([df, pd.DataFrame([{
    'file': new_name, 'top1_label': top3[0][1], 'top1_confidence': round(float(top3[0][2]), 4),
    'actual': orig['actual'], 'correct': new_correct, 'condition': condition}])], ignore_index=True)
display(df)
df.to_csv(CSV, index=False, encoding='utf-8-sig')
files.download(CSV)

## 마무리
- 답안지의 **문제 5(AI 핵심 용어)** 는 노트북 실행 없이 답안지에 바로 작성합니다.
- [파일 → 다운로드 → .ipynb 다운로드]로 `학번_이름_midterm.ipynb` 를 저장해 제출하세요.